<a href="https://colab.research.google.com/github/zelaneroz/sc4000_chatbot_area/blob/main/ChatBotArena_SC4000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LMSYS - Chatbot Arena Human Preference Predictions
[Kaggle Link](https://www.kaggle.com/competitions/lmsys-chatbot-arena/)

Nanyang Technological University
SC4000: Machine Learning
Course Project

**Members**:
* Albinowski, Wiktor Marcin (N2500433A)
* Chelashaw, Joshua Ruigu  (N2500531J)
* Espanto, Zelan Eroz (N2500107L)
* Skendri, Noussayba (N2500525K)
* Umeh, Sophia Chiarapunam (N2503901B)


**Challenge Description**
* Ensuring LLMS' responses resonate with users is critical for successful interaction.
* Predict which response a user will prefer in these head-to-head battles.

This challenge aligns with the concept of "reward models" or "preference models" in reinforcement learning from human feedback (RLHF). Previous research has identified limitations in directly prompting an existing LLM for preference predictions. These limitations often stem from biases such as favoring responses presented first (position bias), being overly verbose (verbosity bias), or exhibiting self-promotion (self-enhancement bias).



**Dataset Descrption**

Dataset collected from Chatbot Arena, where users chat with two anonymous LLMs and choose the answer they prefer.

**Evaluation**
Evaluated on the log loss between the predicted probabilities and the ground truth values (with 'eps=auto').


The submission file should contain a header and have the following format:
```
 id,winner_model_a,winner_model_b,winner_tie
 136060,0.33,0,33,0.33
 211333,0.33,0,33,0.33
 1233961,0.33,0,33,0.33
 etc
```


# 0. Imports

In [29]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Optional
import ast
import hashlib
import math
import re
import csv
import json
import os
import random

In [19]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [20]:
BASE_PATH = 'MyDrive/Acads/Spring26'
df = pd.read_csv(f'/content/drive/{BASE_PATH}/train.csv')
df.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


#  I. Exploratory Data Analysis (EDA)

**Prelim Questions**
* How many data points are there? --> 57,477

**Additional questions**
* How many models are there and variations per model? 64
* How often are each model preferred?
* How many models tie? and which pairings often tie?
* Is there a way to differentiate the prompts? [morals, mathematics, logical, generate, etc] and is there a preference for a particular model for a particular task


In [21]:
# 1. Model Variations
all_models = pd.concat([df['model_a'], df['model_b']])
print(f"Total Unique Models: {all_models.nunique()}")
print(all_models.unique())

Total Unique Models: 64
['gpt-4-1106-preview' 'koala-13b' 'gpt-3.5-turbo-0613' 'llama-2-13b-chat'
 'vicuna-13b' 'mixtral-8x7b-instruct-v0.1' 'gemini-pro' 'gpt-4-0314'
 'vicuna-7b' 'chatglm3-6b' 'pplx-70b-online' 'mpt-30b-chat'
 'llama2-70b-steerlm-chat' 'claude-1' 'claude-2.1' 'chatglm-6b'
 'claude-instant-1' 'dolly-v2-12b' 'claude-2.0' 'deepseek-llm-67b-chat'
 'openchat-3.5' 'starling-lm-7b-alpha' 'gpt-4-0125-preview'
 'llama-2-7b-chat' 'gpt-4-0613' 'wizardlm-70b' 'stablelm-tuned-alpha-7b'
 'vicuna-33b' 'chatglm2-6b' 'dolphin-2.2.1-mistral-7b' 'llama-2-70b-chat'
 'llama-13b' 'palm-2' 'wizardlm-13b' 'codellama-34b-instruct'
 'gemini-pro-dev-api' 'gpt-3.5-turbo-0314' 'gpt-3.5-turbo-1106'
 'yi-34b-chat' 'oasst-pythia-12b' 'qwen-14b-chat' 'alpaca-13b'
 'qwen1.5-72b-chat' 'gpt-3.5-turbo-0125' 'pplx-7b-online'
 'qwen1.5-4b-chat' 'fastchat-t5-3b' 'solar-10.7b-instruct-v1.0'
 'mistral-medium' 'nous-hermes-2-mixtral-8x7b-dpo' 'zephyr-7b-beta'
 'openhermes-2.5-mistral-7b' 'mistral-7b-instruct' 

In [22]:
# 2. Preference Frequency
model_wins = pd.Series(dtype=int)
for idx, row in df.iterrows():
    if row['winner_model_a'] == 1:
        model_wins[row['model_a']] = model_wins.get(row['model_a'], 0) + 1
    elif row['winner_model_b'] == 1:
        model_wins[row['model_b']] = model_wins.get(row['model_b'], 0) + 1

# 3. Tie Analysis
tie_df = df[df['winner_tie'] == 1].copy()
tie_df['pair'] = tie_df.apply(lambda x: "-".join(sorted([x['model_a'], x['model_b']])), axis=1)
print(tie_df['pair'].value_counts())

# 4. Prompt Classification (Simple Keyword Approach)
def categorize(p):
    p = str(p).lower()
    if any(w in p for w in ['moral', 'ethic', 'right']): return 'Morals'
    if any(w in p for w in ['solve', 'math', 'calculate']): return 'Math'
    if any(w in p for w in ['logic', 'if', 'puzzle']): return 'Logic'
    if any(w in p for w in ['write', 'poem', 'create']): return 'Generation'
    return 'General'

df['category'] = df['prompt'].apply(categorize)

# # 5. Preference per Task
# df['winner'] = df.apply(lambda r: r['model_a'] if r['winner_model_a']==1 else (r['model_b'] if r['winner_model_b']==1 else 'Tie'), axis=1)
# category_pref = df[df['winner'] != 'Tie'].groupby(['category', 'winner']).size().unstack(fill_value=0)

# # Visualization
# category_pref.plot(kind='bar', stacked=True)
# plt.title('Model Preference by Task Category')
# plt.show()

pair
gpt-4-0613-gpt-4-1106-preview                   309
claude-1-claude-2.1                             292
claude-2.1-gpt-4-1106-preview                   250
gpt-4-0314-gpt-4-0613                           236
claude-2.1-gpt-4-0613                           189
                                               ... 
dolphin-2.2.1-mistral-7b-mistral-7b-instruct      1
chatglm2-6b-tulu-2-dpo-70b                        1
pplx-70b-online-vicuna-13b                        1
gpt-3.5-turbo-0125-llama2-70b-steerlm-chat        1
dolphin-2.2.1-mistral-7b-llama-2-7b-chat          1
Name: count, Length: 1172, dtype: int64


## My Understanding of the Problem
\(prompt,response A,response B)→P(A preferred)

**What's asked**
* preference prediction / reward modeling problem --> not just a classification problem, more like Pairwise ranking, Human preference modeling, RLHF reward modeling
* Evaluated using log-loss --> calibrated probabilities matter
* output: z = [z_A, z_B, z_tie]

**Log loss**
- Penalizes overconfidence
ex. prediction of [0.99 0.1 0] is bad. better to say [0.6 0.3 0.1]

**Biases**
* position bias (preference for response A)
* verbosity bias
* self-enhancement bias


## Dealing with Biases
**Positioning Bias**

Response A picked more often so:
* randomly swap A/B during training
* Add symmetric training: train on both (A,B) and (B,A)

**Verbosity Bias**

Longer responses are often preferred
* explore features like response length and token count
* normalize or explicitly model it

**Self-enhancement Bias**

LLMs prefer responses that sound more "AI-like" or confident

**Explore existing HuggingFace models**

# II. Data Preprocessing

In [23]:
from sklearn.model_selection import GroupKFold

In [24]:
import ast
import hashlib
import math
import re
from typing import Optional

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold


# -----------------------------
# 1) Text normalization helpers
# -----------------------------
def safe_literal_eval(x):
    """
    Parse string representations of Python lists safely.
    Returns x unchanged if parsing is not needed or fails.
    """
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if x.startswith("[") and x.endswith("]"):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return x
    return x


def flatten_prompt(prompt_value, sep=" [TURN_SEP] "):
    """
    Convert prompt column into a single string.
    Handles list-of-turns and plain strings.
    """
    parsed = safe_literal_eval(prompt_value)

    if isinstance(parsed, list):
        parts = [str(p).strip() for p in parsed if str(p).strip()]
        return sep.join(parts)

    return str(parsed).strip()


def clean_text(text: Optional[str]) -> str:
    """
    Minimal cleaning:
    - preserve case, punctuation, and wording
    - normalize whitespace
    - remove obvious null-like values
    """
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return ""

    text = str(text)

    # If response was stored like '["..."]', optionally unwrap single-item lists
    parsed = safe_literal_eval(text)
    if isinstance(parsed, list):
        if len(parsed) == 1:
            text = str(parsed[0])
        else:
            text = " ".join(str(x) for x in parsed)

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\s+", " ", text).strip()

    return text


# --------------------------------
# 2) Lightweight feature functions
# --------------------------------
def char_len(text: str) -> int:
    return len(text)


def word_count(text: str) -> int:
    if not text:
        return 0
    return len(re.findall(r"\S+", text))


def est_token_count(text: str) -> int:
    """
    Rough tokenizer-free estimate.
    For English-ish text, 1 token is often ~3.5 to 4.5 chars.
    We use 4 as a simple baseline.
    """
    if not text:
        return 0
    return max(1, math.ceil(len(text) / 4))


# ----------------------------------------
# 3) Target + prompt ID + fold assignment
# ----------------------------------------
def make_target_class(df: pd.DataFrame) -> pd.Series:
    """
    0 = A wins
    1 = B wins
    2 = tie
    """
    conds = [
        df["winner_model_a"].eq(1),
        df["winner_model_b"].eq(1),
        df["winner_tie"].eq(1),
    ]
    choices = [0, 1, 2]
    target = np.select(conds, choices, default=-1)

    if (target == -1).any():
        bad_rows = df.loc[target == -1, ["winner_model_a", "winner_model_b", "winner_tie"]]
        raise ValueError(
            "Found rows with invalid winner encoding. "
            "Each row should have exactly one of winner_model_a, winner_model_b, winner_tie = 1.\n"
            f"Sample bad rows:\n{bad_rows.head()}"
        )

    return pd.Series(target, index=df.index, name="target_class")


def make_prompt_id(prompt_text: str) -> str:
    """
    Stable hash-based prompt ID so the same prompt always maps to the same group.
    """
    return hashlib.md5(prompt_text.encode("utf-8")).hexdigest()


def assign_group_folds(df: pd.DataFrame, n_splits: int = 5, group_col: str = "prompt_id") -> pd.DataFrame:
    """
    GroupKFold ensures all rows from the same prompt stay in the same fold.
    This is crucial once you create swapped/symmetric duplicates.
    """
    df = df.copy()
    df["fold"] = -1

    gkf = GroupKFold(n_splits=n_splits)
    X_dummy = np.zeros(len(df))
    y_dummy = df["target_class"].values
    groups = df[group_col].values

    for fold, (_, val_idx) in enumerate(gkf.split(X_dummy, y_dummy, groups)):
        df.loc[df.index[val_idx], "fold"] = fold

    return df


# -------------------------------------
# 4) Core feature engineering function
# -------------------------------------
def add_text_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Response A features
    df["a_char_len"] = df["response_a_text"].map(char_len)
    df["a_word_count"] = df["response_a_text"].map(word_count)
    df["a_est_token_count"] = df["response_a_text"].map(est_token_count)

    # Response B features
    df["b_char_len"] = df["response_b_text"].map(char_len)
    df["b_word_count"] = df["response_b_text"].map(word_count)
    df["b_est_token_count"] = df["response_b_text"].map(est_token_count)

    # Deltas: signed and absolute
    df["char_len_delta"] = df["a_char_len"] - df["b_char_len"]
    df["word_count_delta"] = df["a_word_count"] - df["b_word_count"]
    df["est_token_count_delta"] = df["a_est_token_count"] - df["b_est_token_count"]

    df["abs_char_len_delta"] = df["char_len_delta"].abs()
    df["abs_word_count_delta"] = df["word_count_delta"].abs()
    df["abs_est_token_count_delta"] = df["est_token_count_delta"].abs()

    return df


# ------------------------------------------
# 5) Symmetric augmentation + random swapping
# ------------------------------------------
def make_symmetric_pairs(df: pd.DataFrame) -> pd.DataFrame:
    """
    For every row, generate:
    - original orientation: (A, B)
    - swapped orientation:  (B, A)

    Labels are swapped accordingly:
    - A win <-> B win
    - tie stays tie
    """
    base = df.copy()
    base["pair_orientation"] = "original"

    swapped = df.copy()
    swapped["pair_orientation"] = "swapped"

    # Swap models
    swapped["model_a"], swapped["model_b"] = df["model_b"], df["model_a"]

    # Swap response text
    swapped["response_a_text"], swapped["response_b_text"] = df["response_b_text"], df["response_a_text"]

    # Swap one-hot labels
    swapped["winner_model_a"], swapped["winner_model_b"] = df["winner_model_b"], df["winner_model_a"]
    swapped["winner_tie"] = df["winner_tie"]

    # Swap target class: 0 <-> 1, 2 stays 2
    swapped["target_class"] = swapped["target_class"].map({0: 1, 1: 0, 2: 2})

    out = pd.concat([base, swapped], ignore_index=True)

    # Row ID for each training instance
    out["example_id"] = (
        out["id"].astype(str)
        + "_"
        + out["pair_orientation"].astype(str)
    )

    return out


def randomly_swap_pairs(
    df: pd.DataFrame,
    swap_prob: float = 0.5,
    random_state: int = 42
) -> pd.DataFrame:
    """
    Randomly swap A and B at row level and update labels/features.
    Use this when you want a stochastic training table rather than deterministic doubling.

    Recommended usage:
    - Either use symmetric pairs
    - Or use random swapping each epoch / preprocessing run
    - Or keep both options configurable
    """
    out = df.copy()
    rng = np.random.default_rng(random_state)
    swap_mask = rng.random(len(out)) < swap_prob

    # Temporary copies
    a_model = out.loc[swap_mask, "model_a"].copy()
    a_resp = out.loc[swap_mask, "response_a_text"].copy()
    a_win = out.loc[swap_mask, "winner_model_a"].copy()
    a_target = out.loc[swap_mask, "target_class"].copy()

    # Swap model columns
    out.loc[swap_mask, "model_a"] = out.loc[swap_mask, "model_b"].values
    out.loc[swap_mask, "model_b"] = a_model.values

    # Swap response text columns
    out.loc[swap_mask, "response_a_text"] = out.loc[swap_mask, "response_b_text"].values
    out.loc[swap_mask, "response_b_text"] = a_resp.values

    # Swap one-hot labels
    out.loc[swap_mask, "winner_model_a"] = out.loc[swap_mask, "winner_model_b"].values
    out.loc[swap_mask, "winner_model_b"] = a_win.values

    # Swap target class
    out.loc[swap_mask, "target_class"] = a_target.map({0: 1, 1: 0, 2: 2}).values

    out["random_swap_applied"] = swap_mask.astype(int)

    return out

In [25]:
# -----------------------------------
# 6) End-to-end preprocessing builder
# -----------------------------------
def build_preference_dataframe(
    df: pd.DataFrame,
    n_splits: int = 5,
    use_symmetric_training: bool = True,
    use_random_swap: bool = False,
    swap_prob: float = 0.5,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Build a future-proof preference modeling DataFrame.

    Output includes:
    - prompt_text
    - response_a_text / response_b_text
    - target_class
    - stable prompt_id
    - fold
    - baseline length features
    - example_id
    - orientation flags
    """

    required_cols = [
        "id", "model_a", "model_b", "prompt", "response_a", "response_b",
        "winner_model_a", "winner_model_b", "winner_tie"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    out = df.copy()

    # Normalize prompt and responses
    out["prompt_text"] = out["prompt"].map(flatten_prompt)
    out["response_a_text"] = out["response_a"].map(clean_text)
    out["response_b_text"] = out["response_b"].map(clean_text)

    # Stable single-label target
    out["target_class"] = make_target_class(out)

    # Stable prompt grouping key
    out["prompt_id"] = out["prompt_text"].map(make_prompt_id)

    # Keep traceability to original sample
    out["source_id"] = out["id"].astype(str)

    # Deterministic symmetric training
    if use_symmetric_training:
        out = make_symmetric_pairs(out)
    else:
        out["pair_orientation"] = "original"
        out["example_id"] = out["id"].astype(str) + "_original"

    # Optional stochastic swapping after that
    # In practice, many teams choose ONE of the two.
    # This is left configurable because you asked for both mechanisms.
    if use_random_swap:
        out = randomly_swap_pairs(
            out,
            swap_prob=swap_prob,
            random_state=random_state
        )
    else:
        out["random_swap_applied"] = 0

    # Recompute features after any swapping
    out = add_text_features(out)

    # Group-aware CV folds using prompt_id
    out = assign_group_folds(out, n_splits=n_splits, group_col="prompt_id")

    # Column order
    preferred_order = [
        "example_id", "source_id", "id", "prompt_id", "fold",
        "pair_orientation", "random_swap_applied",
        "model_a", "model_b",
        "prompt_text", "response_a_text", "response_b_text",
        "winner_model_a", "winner_model_b", "winner_tie", "target_class",
        "a_char_len", "a_word_count", "a_est_ntoken_count",
        "b_char_len", "b_word_count", "b_est_token_count",
        "char_len_delta", "word_count_delta", "est_token_count_delta",
        "abs_char_len_delta", "abs_word_count_delta", "abs_est_token_count_delta",
    ]

    existing_cols = [c for c in preferred_order if c in out.columns]
    remaining_cols = [c for c in out.columns if c not in existing_cols]
    out = out[existing_cols + remaining_cols]

    return out

# III. Model Training - Experimental Matrix

In [31]:
!pip install --force-reinstall transformers mpmath

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.8/567.8 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 959.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [30]:
from __future__ import annotations


from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, log_loss
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoConfig,
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False


# =========================================================
# 1) CONFIGS
# =========================================================

@dataclass
class HyperParams:
    learning_rate: float
    weight_decay: float
    dropout: float
    batch_size: int
    num_epochs: int
    max_length: int
    warmup_ratio: float = 0.05
    gradient_accumulation_steps: int = 1
    mixed_precision: bool = True


@dataclass
class SetupConfig:
    name: str
    model_name: str
    architecture_family: str  # "encoder" or "decoder"
    objective_type: str       # "pairwise" or "regression"
    loss_type: str            # "bradley_terry", "sigmoid_ce", "mse"
    features: List[str]       # e.g. ["prompt", "response", "length_features"]
    hyperparameters: HyperParams
    seed: int = 42
    use_wandb: bool = False
    project_name: str = "reward-modeling"
    output_dir: str = "./outputs"
    target_mode: str = "three_way"  # "three_way" or "binary_a_vs_b"
    peft_mode: Optional[str] = None # placeholder for LoRA/QLoRA if needed


def build_experiment_matrix() -> Dict[str, SetupConfig]:
    return {
        "setup_1": SetupConfig(
            name="setup_1_deberta_pairwise_bt",
            model_name="microsoft/deberta-v3-base",
            architecture_family="encoder",
            objective_type="pairwise",
            loss_type="bradley_terry",
            features=["prompt", "response"],
            hyperparameters=HyperParams(
                learning_rate=2e-5,
                weight_decay=0.01,
                dropout=0.10,
                batch_size=8,
                num_epochs=2,
                max_length=512,
            ),
        ),
        "setup_2": SetupConfig(
            name="setup_2_deberta_pairwise_bt_len",
            model_name="microsoft/deberta-v3-base",
            architecture_family="encoder",
            objective_type="pairwise",
            loss_type="bradley_terry",
            features=["prompt", "response", "length_features"],
            hyperparameters=HyperParams(
                learning_rate=2e-5,
                weight_decay=0.01,
                dropout=0.10,
                batch_size=8,
                num_epochs=2,
                max_length=512,
            ),
        ),
        "setup_3": SetupConfig(
            name="setup_3_deberta_pairwise_sigmoid",
            model_name="microsoft/deberta-v3-base",
            architecture_family="encoder",
            objective_type="pairwise",
            loss_type="sigmoid_ce",
            features=["prompt", "response"],
            hyperparameters=HyperParams(
                learning_rate=1.5e-5,
                weight_decay=0.02,
                dropout=0.15,
                batch_size=8,
                num_epochs=2,
                max_length=512,
            ),
        ),
        "setup_4": SetupConfig(
            name="setup_4_deberta_regression_mse",
            model_name="microsoft/deberta-v3-base",
            architecture_family="encoder",
            objective_type="regression",
            loss_type="mse",
            features=["prompt", "response", "length_features"],
            hyperparameters=HyperParams(
                learning_rate=1e-5,
                weight_decay=0.01,
                dropout=0.10,
                batch_size=8,
                num_epochs=2,
                max_length=512,
            ),
        ),
        "setup_5": SetupConfig(
            name="setup_5_llama_pairwise_bt",
            model_name="meta-llama/Llama-2-7b-hf",  # replace with your accessible decoder model
            architecture_family="decoder",
            objective_type="pairwise",
            loss_type="bradley_terry",
            features=["prompt", "response"],
            hyperparameters=HyperParams(
                learning_rate=2e-5,
                weight_decay=0.01,
                dropout=0.05,
                batch_size=2,
                num_epochs=1,
                max_length=1024,
            ),
            peft_mode="lora",
        ),
        "setup_6": SetupConfig(
            name="setup_6_llama_regression_mse_len",
            model_name="meta-llama/Llama-2-7b-hf",
            architecture_family="decoder",
            objective_type="regression",
            loss_type="mse",
            features=["prompt", "response", "length_features"],
            hyperparameters=HyperParams(
                learning_rate=1e-5,
                weight_decay=0.00,
                dropout=0.05,
                batch_size=2,
                num_epochs=1,
                max_length=1024,
            ),
            peft_mode="lora",
        ),
    }


# =========================================================
# 2) UTILS
# =========================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)


def winner_to_class(row: pd.Series) -> int:
    # 0=A, 1=B, 2=tie
    if row["winner_model_a"] == 1:
        return 0
    if row["winner_model_b"] == 1:
        return 1
    if row["winner_tie"] == 1:
        return 2
    raise ValueError("Invalid target row")


def pairwise_target_value(row: pd.Series) -> float:
    # for pairwise ranking:
    # A wins -> 1.0
    # B wins -> 0.0
    # tie -> 0.5
    if row["winner_model_a"] == 1:
        return 1.0
    if row["winner_model_b"] == 1:
        return 0.0
    if row["winner_tie"] == 1:
        return 0.5
    raise ValueError("Invalid target row")


def regression_reward_targets(row: pd.Series) -> Tuple[float, float]:
    # scalar reward targets for A and B
    # A wins -> (1,0), B wins -> (0,1), tie -> (0.5,0.5)
    if row["winner_model_a"] == 1:
        return 1.0, 0.0
    if row["winner_model_b"] == 1:
        return 0.0, 1.0
    if row["winner_tie"] == 1:
        return 0.5, 0.5
    raise ValueError("Invalid target row")


def estimate_token_count(text: str) -> int:
    return max(1, math.ceil(len(text) / 4))


# =========================================================
# 3) DATASET
# =========================================================

class PreferenceDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        config: SetupConfig,
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.config = config
        self.max_length = config.hyperparameters.max_length
        self.use_length_features = "length_features" in config.features

    def __len__(self) -> int:
        return len(self.df)

    def _format_text(self, prompt: str, response: str) -> str:
        if "prompt" in self.config.features and "response" in self.config.features:
            return f"Prompt:\n{prompt}\n\nResponse:\n{response}"
        if "prompt" in self.config.features and "response" not in self.config.features:
            return f"Prompt:\n{prompt}"
        if "response" in self.config.features and "prompt" not in self.config.features:
            return f"Response:\n{response}"
        return f"Prompt:\n{prompt}\n\nResponse:\n{response}"

    def _length_feats(self, row: pd.Series) -> np.ndarray:
        a_chars = len(row["response_a_text"])
        b_chars = len(row["response_b_text"])
        a_words = len(row["response_a_text"].split())
        b_words = len(row["response_b_text"].split())
        a_tok = estimate_token_count(row["response_a_text"])
        b_tok = estimate_token_count(row["response_b_text"])
        feats = np.array([
            a_chars, b_chars, a_chars - b_chars,
            a_words, b_words, a_words - b_words,
            a_tok, b_tok, a_tok - b_tok,
        ], dtype=np.float32)
        return feats

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        prompt = row["prompt_text"]
        response_a = row["response_a_text"]
        response_b = row["response_b_text"]

        text_a = self._format_text(prompt, response_a)
        text_b = self._format_text(prompt, response_b)

        enc_a = self.tokenizer(
            text_a,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )
        enc_b = self.tokenizer(
            text_b,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        example = {
            "input_ids_a": enc_a["input_ids"],
            "attention_mask_a": enc_a["attention_mask"],
            "input_ids_b": enc_b["input_ids"],
            "attention_mask_b": enc_b["attention_mask"],
            "pairwise_target": pairwise_target_value(row),
            "winner_class": winner_to_class(row),
            "id": row.get("example_id", row["id"]),
        }

        if self.config.objective_type == "regression":
            ra, rb = regression_reward_targets(row)
            example["reward_a"] = ra
            example["reward_b"] = rb

        if self.use_length_features:
            example["length_features"] = self._length_feats(row)

        return example


class PreferenceCollator:
    def __init__(self, tokenizer, use_length_features: bool = False):
        self.tokenizer = tokenizer
        self.use_length_features = use_length_features

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        batch_a = [{"input_ids": x["input_ids_a"], "attention_mask": x["attention_mask_a"]} for x in batch]
        batch_b = [{"input_ids": x["input_ids_b"], "attention_mask": x["attention_mask_b"]} for x in batch]

        pad_a = self.tokenizer.pad(batch_a, padding=True, return_tensors="pt")
        pad_b = self.tokenizer.pad(batch_b, padding=True, return_tensors="pt")

        out = {
            "input_ids_a": pad_a["input_ids"],
            "attention_mask_a": pad_a["attention_mask"],
            "input_ids_b": pad_b["input_ids"],
            "attention_mask_b": pad_b["attention_mask"],
            "pairwise_target": torch.tensor([x["pairwise_target"] for x in batch], dtype=torch.float32),
            "winner_class": torch.tensor([x["winner_class"] for x in batch], dtype=torch.long),
            "id": [x["id"] for x in batch],
        }

        if "reward_a" in batch[0]:
            out["reward_a"] = torch.tensor([x["reward_a"] for x in batch], dtype=torch.float32)
            out["reward_b"] = torch.tensor([x["reward_b"] for x in batch], dtype=torch.float32)

        if self.use_length_features:
            out["length_features"] = torch.tensor(
                np.stack([x["length_features"] for x in batch]),
                dtype=torch.float32,
            )

        return out


# =========================================================
# 4) MODEL
# =========================================================

class RewardBackbone(nn.Module):
    """
    Shared wrapper around encoder/decoder backbones.
    Produces one scalar reward per (prompt, response) input.
    """

    def __init__(self, config: SetupConfig, length_feature_dim: int = 9):
        super().__init__()
        self.setup = config
        hf_cfg = AutoConfig.from_pretrained(config.model_name)
        hf_cfg.hidden_dropout_prob = config.hyperparameters.dropout if hasattr(hf_cfg, "hidden_dropout_prob") else getattr(hf_cfg, "hidden_dropout_prob", 0.1)
        hf_cfg.attention_probs_dropout_prob = config.hyperparameters.dropout if hasattr(hf_cfg, "attention_probs_dropout_prob") else getattr(hf_cfg, "attention_probs_dropout_prob", 0.1)

        self.backbone = AutoModel.from_pretrained(config.model_name, config=hf_cfg)
        hidden_size = getattr(hf_cfg, "hidden_size", None) or getattr(hf_cfg, "dim", None) or getattr(hf_cfg, "d_model", None)
        if hidden_size is None:
            raise ValueError("Could not infer hidden size from backbone config.")

        self.use_length_features = "length_features" in config.features
        head_in = hidden_size + (length_feature_dim if self.use_length_features else 0)

        self.dropout = nn.Dropout(config.hyperparameters.dropout)
        self.reward_head = nn.Sequential(
            nn.Linear(head_in, hidden_size // 2),
            nn.Tanh(),
            nn.Dropout(config.hyperparameters.dropout),
            nn.Linear(hidden_size // 2, 1),
        )

    def _pool(self, outputs, attention_mask):
        # General-purpose masked mean pooling
        hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        return pooled

    def forward_once(self, input_ids, attention_mask, length_features=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self._pool(outputs, attention_mask)
        pooled = self.dropout(pooled)

        if self.use_length_features and length_features is not None:
            pooled = torch.cat([pooled, length_features], dim=-1)

        reward = self.reward_head(pooled).squeeze(-1)
        return reward

    def forward(
        self,
        input_ids_a,
        attention_mask_a,
        input_ids_b,
        attention_mask_b,
        length_features=None,
    ):
        reward_a = self.forward_once(input_ids_a, attention_mask_a, length_features)
        reward_b = self.forward_once(input_ids_b, attention_mask_b, length_features)
        return reward_a, reward_b


# =========================================================
# 5) LOSSES
# =========================================================

def bradley_terry_loss(reward_a, reward_b, target):
    # target in {1.0, 0.5, 0.0}
    # p(a preferred) = sigmoid(r_a - r_b)
    diff = reward_a - reward_b
    prob_a = torch.sigmoid(diff)

    # binary CE for hard outcomes, midpoint penalty for ties
    # tie target = 0.5
    loss = nn.functional.binary_cross_entropy(prob_a, target)
    return loss, prob_a


def sigmoid_cross_entropy_loss(reward_a, reward_b, target):
    # mathematically similar, but kept separate for explicit setup routing
    diff = reward_a - reward_b
    prob_a = torch.sigmoid(diff)
    loss = nn.functional.binary_cross_entropy(prob_a, target)
    return loss, prob_a


def mse_reward_loss(reward_a, reward_b, target_a, target_b):
    loss_a = nn.functional.mse_loss(reward_a, target_a)
    loss_b = nn.functional.mse_loss(reward_b, target_b)
    loss = 0.5 * (loss_a + loss_b)
    prob_a = torch.sigmoid(reward_a - reward_b)
    return loss, prob_a


# =========================================================
# 6) METRICS
# =========================================================

def probs_from_pairwise(prob_a: np.ndarray, tie_calibration: bool = True) -> np.ndarray:
    """
    Convert scalar P(A>B) into 3-way probs [A, B, tie].
    Tie handling is heuristic boilerplate:
    more uncertainty near 0.5 -> higher tie probability.
    """
    if tie_calibration:
        tie_prob = 1.0 - np.abs(prob_a - 0.5) * 2.0
        tie_prob = np.clip(tie_prob * 0.20, 0.0, 0.20)  # cap tie mass for baseline
    else:
        tie_prob = np.zeros_like(prob_a)

    remaining = 1.0 - tie_prob
    p_a = remaining * prob_a
    p_b = remaining * (1.0 - prob_a)

    probs = np.stack([p_a, p_b, tie_prob], axis=1)
    probs = probs / probs.sum(axis=1, keepdims=True)
    return probs


def compute_validation_metrics(y_true_class: np.ndarray, prob_a: np.ndarray) -> Dict[str, float]:
    probs_3 = probs_from_pairwise(prob_a)
    y_pred_class = probs_3.argmax(axis=1)

    acc = accuracy_score(y_true_class, y_pred_class)
    ll = log_loss(y_true_class, probs_3, labels=[0, 1, 2])

    return {
        "val_accuracy": float(acc),
        "val_log_loss": float(ll),
    }


# =========================================================
# 7) TRACKER
# =========================================================

class ExperimentTracker:
    def __init__(
        self,
        output_dir: str,
        project_name: str,
        use_wandb: bool = False,
    ):
        self.output_dir = output_dir
        self.project_name = project_name
        self.use_wandb = use_wandb and WANDB_AVAILABLE
        ensure_dir(output_dir)
        self.summary_csv = os.path.join(output_dir, "experiment_summary.csv")
        self.pred_dir = os.path.join(output_dir, "predictions")
        ensure_dir(self.pred_dir)

        if not os.path.exists(self.summary_csv):
            with open(self.summary_csv, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([
                    "setup_name",
                    "model_name",
                    "objective_type",
                    "loss_type",
                    "val_accuracy",
                    "val_log_loss",
                    "elo_rating",
                ])

    def start_run(self, cfg: SetupConfig):
        if self.use_wandb:
            wandb.init(
                project=self.project_name,
                name=cfg.name,
                config=asdict(cfg),
                reinit=True,
            )

    def log_metrics(self, metrics: Dict[str, float], step: Optional[int] = None):
        if self.use_wandb:
            wandb.log(metrics, step=step)

    def finish_run(self):
        if self.use_wandb:
            wandb.finish()

    def save_predictions(self, setup_name: str, ids: List[Any], y_true: np.ndarray, prob_a: np.ndarray):
        df = pd.DataFrame({
            "id": ids,
            "y_true_class": y_true,
            "prob_a_pref": prob_a,
        })
        path = os.path.join(self.pred_dir, f"{setup_name}_val_predictions.csv")
        df.to_csv(path, index=False)

    def append_summary(self, row: Dict[str, Any]):
        with open(self.summary_csv, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                row["setup_name"],
                row["model_name"],
                row["objective_type"],
                row["loss_type"],
                row["val_accuracy"],
                row["val_log_loss"],
                row.get("elo_rating", np.nan),
            ])


def compute_elo_from_prediction_files(pred_dir: str, k_factor: float = 32.0) -> Dict[str, float]:
    """
    Elo over setups, based on per-example negative log-likelihood.
    For each pair of setups, on each shared example:
    - lower NLL gets a 'win'
    - equal gets a draw
    """
    files = sorted(Path(pred_dir).glob("*_val_predictions.csv"))
    if len(files) < 2:
        return {}

    preds = {}
    for fp in files:
        name = fp.stem.replace("_val_predictions", "")
        df = pd.read_csv(fp)
        probs3 = probs_from_pairwise(df["prob_a_pref"].values)
        # per-example NLL
        y = df["y_true_class"].values.astype(int)
        nll = -np.log(np.clip(probs3[np.arange(len(y)), y], 1e-15, 1.0))
        preds[name] = pd.DataFrame({"id": df["id"], "nll": nll})

    ratings = {name: 1500.0 for name in preds.keys()}

    names = list(preds.keys())
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            merged = preds[a].merge(preds[b], on="id", suffixes=("_a", "_b"))
            for _, row in merged.iterrows():
                ra, rb = ratings[a], ratings[b]
                ea = 1 / (1 + 10 ** ((rb - ra) / 400))
                eb = 1 - ea

                if row["nll_a"] < row["nll_b"]:
                    sa, sb = 1.0, 0.0
                elif row["nll_a"] > row["nll_b"]:
                    sa, sb = 0.0, 1.0
                else:
                    sa, sb = 0.5, 0.5

                ratings[a] = ra + k_factor * (sa - ea)
                ratings[b] = rb + k_factor * (sb - eb)

    return ratings


# =========================================================
# 8) TRAIN / EVAL
# =========================================================

def build_optimizer_and_scheduler(model, cfg: SetupConfig, total_steps: int):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.hyperparameters.learning_rate,
        weight_decay=cfg.hyperparameters.weight_decay,
    )
    warmup_steps = int(total_steps * cfg.hyperparameters.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    return optimizer, scheduler


def train_one_epoch(model, loader, optimizer, scheduler, device, cfg, tracker=None, epoch=0):
    model.train()
    total_loss = 0.0

    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.hyperparameters.mixed_precision and torch.cuda.is_available()))

    for step, batch in enumerate(loader):
        input_ids_a = batch["input_ids_a"].to(device)
        attention_mask_a = batch["attention_mask_a"].to(device)
        input_ids_b = batch["input_ids_b"].to(device)
        attention_mask_b = batch["attention_mask_b"].to(device)

        length_features = batch.get("length_features")
        if length_features is not None:
            length_features = length_features.to(device)

        with torch.cuda.amp.autocast(enabled=(cfg.hyperparameters.mixed_precision and torch.cuda.is_available())):
            reward_a, reward_b = model(
                input_ids_a=input_ids_a,
                attention_mask_a=attention_mask_a,
                input_ids_b=input_ids_b,
                attention_mask_b=attention_mask_b,
                length_features=length_features,
            )

            if cfg.objective_type == "pairwise":
                target = batch["pairwise_target"].to(device)
                if cfg.loss_type == "bradley_terry":
                    loss, _ = bradley_terry_loss(reward_a, reward_b, target)
                elif cfg.loss_type == "sigmoid_ce":
                    loss, _ = sigmoid_cross_entropy_loss(reward_a, reward_b, target)
                else:
                    raise ValueError(f"Unsupported pairwise loss: {cfg.loss_type}")

            elif cfg.objective_type == "regression":
                target_a = batch["reward_a"].to(device)
                target_b = batch["reward_b"].to(device)
                if cfg.loss_type == "mse":
                    loss, _ = mse_reward_loss(reward_a, reward_b, target_a, target_b)
                else:
                    raise ValueError(f"Unsupported regression loss: {cfg.loss_type}")

            else:
                raise ValueError(f"Unsupported objective_type: {cfg.objective_type}")

            loss = loss / cfg.hyperparameters.gradient_accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % cfg.hyperparameters.gradient_accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        total_loss += loss.item()

        if tracker is not None:
            tracker.log_metrics({"train_loss": loss.item()}, step=epoch * len(loader) + step)

    return total_loss / max(len(loader), 1)


@torch.no_grad()
def evaluate(model, loader, device, cfg):
    model.eval()

    all_ids = []
    all_targets = []
    all_prob_a = []
    total_loss = 0.0

    for batch in loader:
        input_ids_a = batch["input_ids_a"].to(device)
        attention_mask_a = batch["attention_mask_a"].to(device)
        input_ids_b = batch["input_ids_b"].to(device)
        attention_mask_b = batch["attention_mask_b"].to(device)

        length_features = batch.get("length_features")
        if length_features is not None:
            length_features = length_features.to(device)

        reward_a, reward_b = model(
            input_ids_a=input_ids_a,
            attention_mask_a=attention_mask_a,
            input_ids_b=input_ids_b,
            attention_mask_b=attention_mask_b,
            length_features=length_features,
        )

        if cfg.objective_type == "pairwise":
            target = batch["pairwise_target"].to(device)
            if cfg.loss_type == "bradley_terry":
                loss, prob_a = bradley_terry_loss(reward_a, reward_b, target)
            elif cfg.loss_type == "sigmoid_ce":
                loss, prob_a = sigmoid_cross_entropy_loss(reward_a, reward_b, target)
            else:
                raise ValueError(f"Unsupported pairwise loss: {cfg.loss_type}")

        elif cfg.objective_type == "regression":
            target_a = batch["reward_a"].to(device)
            target_b = batch["reward_b"].to(device)
            if cfg.loss_type == "mse":
                loss, prob_a = mse_reward_loss(reward_a, reward_b, target_a, target_b)
            else:
                raise ValueError(f"Unsupported regression loss: {cfg.loss_type}")

        total_loss += loss.item()

        all_ids.extend(batch["id"])
        all_targets.extend(batch["winner_class"].cpu().numpy().tolist())
        all_prob_a.extend(prob_a.detach().cpu().numpy().tolist())

    y_true = np.array(all_targets)
    prob_a = np.array(all_prob_a)

    metrics = compute_validation_metrics(y_true, prob_a)
    metrics["val_objective_loss"] = float(total_loss / max(len(loader), 1))

    return metrics, all_ids, y_true, prob_a


def train_reward_model(
    config: SetupConfig,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
) -> Dict[str, Any]:
    set_seed(config.seed)
    ensure_dir(config.output_dir)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_ds = PreferenceDataset(train_df, tokenizer, config)
    val_ds = PreferenceDataset(val_df, tokenizer, config)
    collator = PreferenceCollator(tokenizer, use_length_features=("length_features" in config.features))

    train_loader = DataLoader(
        train_ds,
        batch_size=config.hyperparameters.batch_size,
        shuffle=True,
        collate_fn=collator,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config.hyperparameters.batch_size,
        shuffle=False,
        collate_fn=collator,
    )

    model = RewardBackbone(config).to(device)

    total_steps = math.ceil(len(train_loader) * config.hyperparameters.num_epochs / config.hyperparameters.gradient_accumulation_steps)
    optimizer, scheduler = build_optimizer_and_scheduler(model, config, total_steps)

    tracker = ExperimentTracker(
        output_dir=config.output_dir,
        project_name=config.project_name,
        use_wandb=config.use_wandb,
    )
    tracker.start_run(config)

    best_metrics = None
    best_state = None

    for epoch in range(config.hyperparameters.num_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device, config, tracker, epoch)
        val_metrics, ids, y_true, prob_a = evaluate(model, val_loader, device, config)

        epoch_metrics = {
            "epoch": epoch,
            "train_epoch_loss": train_loss,
            **val_metrics,
        }
        tracker.log_metrics(epoch_metrics, step=(epoch + 1) * len(train_loader))

        if best_metrics is None or val_metrics["val_log_loss"] < best_metrics["val_log_loss"]:
            best_metrics = val_metrics
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            tracker.save_predictions(config.name, ids, y_true, prob_a)

    ckpt_path = os.path.join(config.output_dir, f"{config.name}_best.pt")
    if best_state is not None:
        torch.save(best_state, ckpt_path)

    tracker.finish_run()

    return {
        "setup_name": config.name,
        "model_name": config.model_name,
        "objective_type": config.objective_type,
        "loss_type": config.loss_type,
        **best_metrics,
        "checkpoint_path": ckpt_path,
    }


# =========================================================
# 9) EXPERIMENT ORCHESTRATION
# =========================================================

def run_experiment_matrix(
    processed_df: pd.DataFrame,
    setups: Dict[str, SetupConfig],
    fold: int = 0,
) -> pd.DataFrame:
    """
    Assumes processed_df already has:
    - prompt_text
    - response_a_text
    - response_b_text
    - fold
    - example_id or id
    - winner_model_a / winner_model_b / winner_tie
    """
    results = []

    for _, cfg in setups.items():
        print(f"\n==== Running {cfg.name} ====")
        train_df = processed_df[processed_df["fold"] != fold].copy()
        val_df = processed_df[processed_df["fold"] == fold].copy()

        summary = train_reward_model(cfg, train_df, val_df)
        results.append(summary)

    # Compute Elo across all saved validation predictions
    pred_dir = next(iter(setups.values())).output_dir + "/predictions"
    elo_ratings = compute_elo_from_prediction_files(pred_dir)

    final_rows = []
    tracker = ExperimentTracker(
        output_dir=next(iter(setups.values())).output_dir,
        project_name=next(iter(setups.values())).project_name,
        use_wandb=False,
    )

    for row in results:
        row["elo_rating"] = float(elo_ratings.get(row["setup_name"], 1500.0))
        tracker.append_summary(row)
        final_rows.append(row)

    result_df = pd.DataFrame(final_rows).sort_values(
        ["val_log_loss", "elo_rating"],
        ascending=[True, False],
    ).reset_index(drop=True)

    result_df.to_csv(
        os.path.join(next(iter(setups.values())).output_dir, "final_comparison.csv"),
        index=False,
    )
    return result_df


# =========================================================
# 10) EXAMPLE ENTRYPOINT
# =========================================================

if __name__ == "__main__":
    # Example:
    # processed_df = pd.read_csv("./processed_preference_df.csv")
    # Must include the fold column and cleaned text columns from preprocessing.
    #
    # Minimal expected columns:
    # ['id', 'prompt_text', 'response_a_text', 'response_b_text',
    #  'winner_model_a', 'winner_model_b', 'winner_tie', 'fold']

    processed_df = pd.read_csv("./processed_preference_df.csv")
    setups = build_experiment_matrix()

    # global defaults for output / tracking
    for _, cfg in setups.items():
        cfg.output_dir = "./reward_model_runs"
        cfg.use_wandb = False  # set True if wandb is installed and configured
        cfg.project_name = "arena-preference-reward-modeling"

    final_comparison = run_experiment_matrix(
        processed_df=processed_df,
        setups=setups,
        fold=0,
    )

    print("\nFinal comparison:")
    print(final_comparison)

FileNotFoundError: [Errno 2] No such file or directory: './processed_preference_df.csv'

In [ ]:
# setups = build_experiment_matrix()

# setups = {k: v for k, v in setups.items() if "llama" not in v.name}

# for cfg in setups.values():
#     cfg.output_dir = "/content/drive/MyDrive/YOUR_FOLDER_NAME/runs"
#     cfg.hyperparameters.batch_size = 4

# results = run_experiment_matrix(df, setups, fold=0)
# results

# IV. Model Comparison

In [26]:
def run_single_setup_across_folds(processed_df, cfg, train_reward_model_fn, folds=None):
    """
    Runs one setup across all requested folds.
    Returns:
        fold_results_df: one row per fold
        summary_row: aggregated metrics for comparison table
    """
    if folds is None:
        folds = sorted(processed_df["fold"].unique().tolist())

    fold_rows = []

    for fold in folds:
        train_df = processed_df[processed_df["fold"] != fold].copy()
        val_df = processed_df[processed_df["fold"] == fold].copy()

        result = train_reward_model_fn(cfg, train_df, val_df)

        row = {
            "setup_name": cfg.name,
            "model_name": cfg.model_name,
            "objective_type": cfg.objective_type,
            "loss_type": cfg.loss_type,
            "fold": fold,
            "val_accuracy": result.get("val_accuracy", np.nan),
            "val_log_loss": result.get("val_log_loss", np.nan),
            "elo_rating": result.get("elo_rating", np.nan),
            "checkpoint_path": result.get("checkpoint_path", None),
        }
        fold_rows.append(row)

    fold_results_df = pd.DataFrame(fold_rows)

    summary_row = {
        "setup_name": cfg.name,
        "model_name": cfg.model_name,
        "objective_type": cfg.objective_type,
        "loss_type": cfg.loss_type,
        "num_folds": len(fold_results_df),
        "mean_val_accuracy": fold_results_df["val_accuracy"].mean(),
        "std_val_accuracy": fold_results_df["val_accuracy"].std(ddof=0),
        "mean_val_log_loss": fold_results_df["val_log_loss"].mean(),
        "std_val_log_loss": fold_results_df["val_log_loss"].std(ddof=0),
        "mean_elo_rating": fold_results_df["elo_rating"].mean(skipna=True),
        "std_elo_rating": fold_results_df["elo_rating"].std(ddof=0, skipna=True),
    }

    return fold_results_df, summary_row


def build_model_comparison_table(
    processed_df,
    setups,
    train_reward_model_fn,
    folds=None,
    output_dir="./comparison_outputs"
):
    """
    Executes the full experimental matrix:
    - loops over setups
    - loops over folds
    - collects per-fold metrics
    - builds aggregated model comparison table
    """
    os.makedirs(output_dir, exist_ok=True)

    all_fold_results = []
    summary_rows = []

    for setup_key, cfg in setups.items():
        print(f"\nRunning {setup_key}: {cfg.name}")

        fold_df, summary_row = run_single_setup_across_folds(
            processed_df=processed_df,
            cfg=cfg,
            train_reward_model_fn=train_reward_model_fn,
            folds=folds,
        )

        all_fold_results.append(fold_df)
        summary_rows.append(summary_row)

    all_fold_results_df = pd.concat(all_fold_results, ignore_index=True)
    comparison_df = pd.DataFrame(summary_rows)

    # sort: lower log loss is better, higher accuracy is better
    comparison_df = comparison_df.sort_values(
        by=["mean_val_log_loss", "mean_val_accuracy"],
        ascending=[True, False]
    ).reset_index(drop=True)

    # add rank columns
    comparison_df["rank_by_log_loss"] = comparison_df["mean_val_log_loss"].rank(method="min", ascending=True)
    comparison_df["rank_by_accuracy"] = comparison_df["mean_val_accuracy"].rank(method="min", ascending=False)
    comparison_df["overall_rank"] = comparison_df[["rank_by_log_loss", "rank_by_accuracy"]].mean(axis=1)
    comparison_df = comparison_df.sort_values(
        by=["overall_rank", "mean_val_log_loss"],
        ascending=[True, True]
    ).reset_index(drop=True)

    # save outputs
    fold_path = os.path.join(output_dir, "all_fold_results.csv")
    comparison_path = os.path.join(output_dir, "model_comparison_table.csv")

    all_fold_results_df.to_csv(fold_path, index=False)
    comparison_df.to_csv(comparison_path, index=False)

    return all_fold_results_df, comparison_df


def print_leaderboard(comparison_df, top_k=None):
    display_cols = [
        "setup_name",
        "model_name",
        "objective_type",
        "loss_type",
        "mean_val_log_loss",
        "std_val_log_loss",
        "mean_val_accuracy",
        "std_val_accuracy",
        "mean_elo_rating",
        "overall_rank",
    ]

    view = comparison_df[display_cols].copy()
    if top_k is not None:
        view = view.head(top_k)

    print("\n=== Model Comparison Table ===")
    print(view.to_string(index=False))


# -------------------------
# Example usage
# -------------------------
# Assumes you already have:
# 1. processed_df from your preprocessing pipeline
# 2. setups from build_experiment_matrix()
# 3. train_reward_model(config, train_df, val_df)

# setups = build_experiment_matrix()
# all_fold_results_df, comparison_df = build_model_comparison_table(
#     processed_df=processed_df,
#     setups=setups,
#     train_reward_model_fn=train_reward_model,
#     folds=[0, 1, 2, 3, 4],   # or None to use all folds in processed_df
#     output_dir="./comparison_outputs",
# )
# print_leaderboard(comparison_df, top_k=10)